# FIFA World Cup 2026 — Notebook 1: Data & Feature Pipeline (v2)

**What changed from v1:**
- Training data expanded from ~964 WC matches to **49,000+ international matches**
- Elo ratings computed from full international history (rolling, tournament-weighted)
- Pre-tournament form from last 18 months of all international matches
- Name normalisation map to prevent mismatches between `martj42` and `jfjelstul` naming
- Diagnostic prints flag any 2026 WC team that falls back to league-average Elo

**Data sources (both auto-scraped):**
- `martj42/international_results` — 49,445 international matches 1872–2026
- `jfjelstul/worldcup` — detailed WC data: goals, squads, awards, player appearances

**Outputs:** `team_features.csv`, `player_features.csv`

## 1. Setup

In [ ]:
!pip install requests pandas numpy matplotlib seaborn scikit-learn scipy --quiet

import requests, io, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")
print("Ready")

## 2. Name Normalisation Map

The two data sources (`martj42` and `jfjelstul`) use different team name conventions.
All downstream merges go through this map so teams are never silently dropped or rated as unknowns.

In [ ]:
# Maps martj42 names → jfjelstul names (jfjelstul is the canonical side)
# Also used to normalise team names in the 2026 group draw
NAME_NORM = {
    # Special characters / diacritics
    "Türkiye":                      "Turkey",
    "Ivory Coast":                  "Côte d'Ivoire",
    "Cote d'Ivoire":                "Côte d'Ivoire",
    "Cape Verde":                   "Cabo Verde",
    "Cape Verde Islands":           "Cabo Verde",
    "DR Congo":                     "Democratic Republic of the Congo",
    "Congo DR":                     "Democratic Republic of the Congo",
    "Republic of Ireland":          "Ireland",
    "Korea Republic":               "South Korea",
    "Korea DPR":                    "North Korea",
    "Chinese Taipei":               "Taiwan",
    "China PR":                     "China",
    "USA":                          "United States",
    "Bosnia-Herzegovina":           "Bosnia and Herzegovina",
    # Historical/alternate spellings common in martj42
    "West Germany":                 "Germany",
    "Czechoslovakia":               "Czech Republic",   # post-split, assign to CZE
    "Yugoslavia":                   "Serbia",           # approximate
    "Soviet Union":                 "Russia",           # approximate
}

def normalise(name):
    return NAME_NORM.get(name, name)

print(f"Name normalisation map loaded: {len(NAME_NORM)} entries")
print("These names will be standardised before any merge or Elo lookup.")

## 3. Scrape Raw Data

In [ ]:
INTL_BASE = "https://raw.githubusercontent.com/martj42/international_results/master/"
WC_BASE   = "https://raw.githubusercontent.com/jfjelstul/worldcup/master/data-csv/"

def fetch(url, label):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), low_memory=False)
    print(f"  {label:<40} {df.shape[0]:>6,} rows x {df.shape[1]} cols")
    return df

print("Fetching international results...")
intl_results    = fetch(INTL_BASE + "results.csv",      "results (all international)")
intl_scorers    = fetch(INTL_BASE + "goalscorers.csv",  "goalscorers")

print("\nFetching World Cup detail data...")
wc_matches      = fetch(WC_BASE + "matches.csv",           "wc_matches")
wc_goals        = fetch(WC_BASE + "goals.csv",             "wc_goals")
wc_squads       = fetch(WC_BASE + "squads.csv",            "wc_squads")
wc_players      = fetch(WC_BASE + "players.csv",           "wc_players")
wc_tournaments  = fetch(WC_BASE + "tournaments.csv",       "wc_tournaments")
wc_team_app     = fetch(WC_BASE + "team_appearances.csv",  "wc_team_appearances")
wc_player_app   = fetch(WC_BASE + "player_appearances.csv","wc_player_appearances")
wc_award_winners= fetch(WC_BASE + "award_winners.csv",     "wc_award_winners")
print("\nAll files loaded.")

## 4. Reference Maps

In [ ]:
men_tourns  = wc_tournaments[wc_tournaments["tournament_name"].str.contains("Men", na=False)]
YEAR_MAP    = men_tourns.set_index("tournament_id")["year"].to_dict()
WINNER_MAP  = men_tourns.set_index("tournament_id")["winner"].to_dict()
HOST_MAP    = men_tourns.set_index("year")["host_country"].to_dict()
WC_YEARS    = sorted(men_tourns["year"].astype(int).tolist())

STAGE_RANK = {
    "group stage": 1,       "second group stage": 2,
    "final round": 2,       "round of 16": 3,
    "quarter-final": 4,     "quarter-finals": 4,
    "semi-final": 5,        "semi-finals": 5,
    "third-place match": 6, "final": 7,
}

AWARD_IDS = {
    "A-1": "won_best_player",
    "A-4": "won_golden_boot",
    "A-8": "won_young_player",
}

# Tournament importance weights for Elo K-factor
# WC=60 because a WC match has ~3x the competitive stakes of a friendly.
# Confederal tournaments (Euros, Copa) ≈ 50; qualifiers slightly less.
# Friendlies=20 as low-stakes floor.
# k=15 shrinkage in Poisson is separate — intentionally conservative to avoid
# overfitting small WC samples (22 tournaments). More data helps Elo but the
# Poisson rating pool still only has ~22 labelled outcomes per award category.
K_MAP = {
    "FIFA World Cup":                    60,
    "FIFA World Cup qualification":      40,
    "UEFA Euro":                         50,
    "UEFA Euro qualification":           35,
    "Copa America":                      50,
    "Copa América":                      50,
    "African Cup of Nations":            50,
    "African Cup of Nations qualification": 35,
    "AFC Asian Cup":                     45,
    "AFC Asian Cup qualification":       35,
    "UEFA Nations League":               40,
    "CONCACAF Nations League":           40,
    "Gold Cup":                          40,
    "CONCACAF Gold Cup":                 40,
    "Friendly":                          20,
}
DEFAULT_K = 30

def men_wc_only(df):
    df = df.copy()
    df["year"] = df["tournament_id"].map(YEAR_MAP)
    return df[df["year"].notna()].reset_index(drop=True)

# Filter WC detail tables
m_matches    = men_wc_only(wc_matches)
m_goals      = men_wc_only(wc_goals)
m_squads     = men_wc_only(wc_squads)
m_team_app   = men_wc_only(wc_team_app)
m_player_app = men_wc_only(wc_player_app)
m_awards     = men_wc_only(wc_award_winners)

# Clean international results + apply name normalisation
intl = intl_results.copy()
intl["date"]       = pd.to_datetime(intl["date"])
intl["year"]       = intl["date"].dt.year
intl["home_score"] = pd.to_numeric(intl["home_score"], errors="coerce")
intl["away_score"] = pd.to_numeric(intl["away_score"], errors="coerce")
intl = intl.dropna(subset=["home_score","away_score"]).reset_index(drop=True)

# Normalise team names in the international results dataset
intl["home_team"] = intl["home_team"].map(normalise)
intl["away_team"] = intl["away_team"].map(normalise)

print(f"WC matches (men's)    : {len(m_matches):,}")
print(f"International results : {len(intl):,}  ({intl['date'].min().date()} to {intl['date'].max().date()})")
print(f"WC years covered      : {WC_YEARS[-5:]}")

## 5. Rolling Elo Ratings (from 49k international matches)

**Key design decisions:**
- `from_year=1990`: modern era cutoff — pre-1990 data has sparse coverage and historical political teams (USSR, Yugoslavia, Czechoslovakia) that distort ratings
- Tournament-weighted K-factor: WC matches move ratings 3x more than friendlies
- Margin-of-victory multiplier via `log1p(|goal diff|) + 1`
- Snapshot taken *before* each WC starts → no data leakage
- Final snapshot at end of all data = 2026 pre-tournament ratings

In [ ]:
def compute_rolling_elo(df, initial=1500, from_year=1990):
    df = df[df["year"] >= from_year].sort_values("date").copy()
    elo = {}

    def get(t): return elo.get(t, initial)
    def exp_s(a, b): return 1 / (1 + 10**((b-a)/400))

    wc_start_dates = (
        men_tourns
        .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
        .set_index("year")["start_date"]
        .to_dict()
    )
    snapshots_taken = set()
    elo_snapshots   = {}

    for _, row in df.iterrows():
        h, a   = row["home_team"], row["away_team"]
        hg, ag = row["home_score"], row["away_score"]
        tourn  = row.get("tournament","")
        date   = row["date"]
        k      = K_MAP.get(tourn, DEFAULT_K)

        for wc_yr, wc_date in wc_start_dates.items():
            if pd.notna(wc_date) and wc_yr not in snapshots_taken:
                if date >= wc_date:
                    all_teams = set(df["home_team"]) | set(df["away_team"])
                    for team in all_teams:
                        elo_snapshots[(int(wc_yr), team)] = round(get(team), 1)
                    snapshots_taken.add(wc_yr)

        eh    = exp_s(get(h), get(a))
        act_h = 1.0 if hg > ag else (0.5 if hg == ag else 0.0)
        mov   = np.log1p(abs(hg-ag)) + 1

        elo[h] = get(h) + k * mov * (act_h - eh)
        elo[a] = get(a) + k * mov * ((1-act_h) - (1-eh))

    # Final snapshot → 2026 pre-tournament ratings
    all_teams = set(df["home_team"]) | set(df["away_team"])
    for team in all_teams:
        elo_snapshots[(2026, team)] = round(get(team), 1)

    return elo, elo_snapshots

print("Computing Elo from international results (this takes ~30 seconds)...")
elo_current, elo_snapshots = compute_rolling_elo(intl)

elo_df = pd.DataFrame([
    {"year": yr, "team": team, "elo": val}
    for (yr, team), val in elo_snapshots.items()
])

print(f"Elo computed for {len(elo_snapshots):,} (year, team) pairs")
print("\nCurrent Elo rankings (going into 2026):")
elo_2026 = (elo_df[elo_df["year"]==2026]
            .sort_values("elo", ascending=False)
            .reset_index(drop=True))
display(elo_2026.head(20))

## 6. Diagnostic — 2026 WC Teams Without Elo History

Any 2026 WC team that appears here genuinely lacks data in `martj42`.
Teams that *should* have history but appear here indicate a name-mismatch bug — fix `NAME_NORM` in Section 2.

In [ ]:
GROUPS_2026 = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],      # Türkiye normalised
    "E": ["Germany", "Curaçao", "Côte d'Ivoire", "Ecuador"],        # normalised
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],        # normalised
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Democratic Republic of the Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

all_2026_teams = [t for teams in GROUPS_2026.values() for t in teams]
elo_2026_lookup = elo_df[elo_df["year"]==2026].set_index("team")["elo"].to_dict()
AVG_ELO = elo_df[elo_df["year"]==2026]["elo"].mean()

print("=== 2026 WC Team Elo Coverage ===")
missing = []
for team in sorted(all_2026_teams):
    elo_val = elo_2026_lookup.get(team)
    if elo_val is None:
        missing.append(team)
        print(f"  MISSING  {team:<40} → will use league average ({AVG_ELO:.0f})")
    else:
        flag = " ✓" if elo_val > AVG_ELO else "  "
        print(f"  {flag}  {team:<40} Elo = {elo_val:.0f}")

print(f"\n{len(missing)} teams without Elo history (will use league average)")
if missing:
    print("\nACTION: If any of these look wrong (e.g. Scotland, DR Congo),")
    print("add the correct martj42 name to NAME_NORM in Section 2 and re-run.")

## 7. Pre-Tournament Form Features

In [ ]:
def compute_form_features(intl_df, wc_tournaments_df, n_matches=20):
    t_dates = (
        wc_tournaments_df[wc_tournaments_df["tournament_name"].str.contains("Men", na=False)]
        .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
        .set_index("year")[["start_date","host_country"]]
    )

    records = []
    for wc_year, row in t_dates.iterrows():
        wc_start = row["start_date"]
        if pd.isna(wc_start): continue
        wc_year = int(wc_year)

        cutoff = wc_start - pd.DateOffset(months=18)
        pre_wc = intl_df[
            (intl_df["date"] >= cutoff) &
            (intl_df["date"] < wc_start)
        ].copy()

        teams = set(pre_wc["home_team"]) | set(pre_wc["away_team"])
        for team in teams:
            h = pre_wc[pre_wc["home_team"] == team]
            a = pre_wc[pre_wc["away_team"] == team]

            all_matches = pd.concat([
                h.assign(gf=h["home_score"], ga=h["away_score"]),
                a.assign(gf=a["away_score"], ga=a["home_score"]),
            ]).sort_values("date").tail(n_matches)

            if len(all_matches) == 0: continue

            wins   = ((all_matches["gf"] > all_matches["ga"])).sum()
            draws  = ((all_matches["gf"] == all_matches["ga"])).sum()
            played = len(all_matches)
            gf     = all_matches["gf"].sum()
            ga     = all_matches["ga"].sum()

            records.append({
                "year":                 wc_year,
                "team_name":            team,
                "form_matches":         played,
                "form_win_rate":        wins / played,
                "form_draw_rate":       draws / played,
                "form_gd_per_game":     (gf - ga) / played,
                "form_gf_per_game":     gf / played,
                "form_ga_per_game":     ga / played,
                "form_points_per_game": (3*wins + draws) / played,
            })

    return pd.DataFrame(records)

form_df = compute_form_features(intl, wc_tournaments)
print(f"Form features: {len(form_df):,} rows")
print("\nSample - 2022 top form teams:")
display(form_df[form_df["year"]==2022]
        .sort_values("form_points_per_game", ascending=False)
        .head(8)[["team_name","form_matches","form_win_rate","form_gd_per_game","form_points_per_game"]])

## 8. Squad Profile Features

In [ ]:
def compute_squad_features(squads_df, players_df, tournaments_df):
    sq = squads_df.copy()
    sq["year"] = sq["year"].astype(int)

    bio = players_df[["player_id","birth_date","list_tournaments"]].copy()
    bio["birth_date"] = pd.to_datetime(bio["birth_date"], errors="coerce")
    sq = sq.merge(bio, on="player_id", how="left")

    t_starts = (
        tournaments_df[tournaments_df["tournament_name"].str.contains("Men", na=False)]
        .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
        .set_index("year")["start_date"].to_dict()
    )
    sq["t_start"] = sq["year"].map(t_starts)
    sq["age"] = ((sq["t_start"] - sq["birth_date"]).dt.days / 365.25)

    def prev_wc(row):
        try:
            yrs = [int(y.strip()) for y in str(row["list_tournaments"]).split(",")]
            return sum(1 for y in yrs if y < int(row["year"]))
        except:
            return 0
    sq["wc_exp"] = sq.apply(prev_wc, axis=1)

    agg = sq.groupby(["year","team_name"]).agg(
        squad_size    = ("player_id",  "count"),
        squad_avg_age = ("age",        "mean"),
        squad_min_age = ("age",        "min"),
        squad_avg_exp = ("wc_exp",     "mean"),
    ).reset_index()

    for pos, col in [("goalkeeper","n_gk"),("defender","n_def"),
                      ("midfielder","n_mid"),("forward","n_fwd")]:
        cnt = (sq[sq["position_name"].str.lower().str.contains(pos, na=False)]
               .groupby(["year","team_name"]).size().reset_index(name=col))
        agg = agg.merge(cnt, on=["year","team_name"], how="left")
        agg[col] = agg[col].fillna(0).astype(int)

    agg["squad_avg_age"] = agg["squad_avg_age"].round(1)
    agg["squad_avg_exp"] = agg["squad_avg_exp"].round(2)
    return agg

squad_df = compute_squad_features(m_squads, wc_players, wc_tournaments)
print(f"Squad features: {len(squad_df):,} rows")

## 9. Tournament History Features

In [ ]:
def compute_history(team_app_df):
    df = team_app_df.copy()
    df["year"]       = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)

    max_stage = (df.groupby(["year","team_name"])["stage_rank"]
                 .max().reset_index()
                 .sort_values(["team_name","year"]))

    records = []
    for team, grp in max_stage.groupby("team_name"):
        grp = grp.sort_values("year").reset_index(drop=True)
        for i, row in grp.iterrows():
            past = grp[grp["year"] < row["year"]]
            records.append({
                "year":            row["year"],
                "team_name":       team,
                "wc_appearances":  len(past),
                "best_stage_ever": past["stage_rank"].max() if len(past) > 0 else 0,
            })

    hist = pd.DataFrame(records)
    hist["is_host"]     = hist.apply(
        lambda r: int(HOST_MAP.get(int(r["year"]),"") == r["team_name"]), axis=1)
    hist["won_last_wc"] = hist.apply(
        lambda r: int(WINNER_MAP.get(f"WC-{int(r['year'])-4}","") == r["team_name"]), axis=1)
    hist["best_stage_ever"] = hist["best_stage_ever"].fillna(0)
    return hist

history_df = compute_history(m_team_app)
print(f"History features: {len(history_df):,} rows")

## 10. Merge All Features → team_features

In [ ]:
def compute_targets(team_app_df):
    df = team_app_df.copy()
    df["year"]       = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)
    targets = df.groupby(["year","team_name"])["stage_rank"].max().reset_index()
    targets["won_tournament"] = targets.apply(
        lambda r: int(WINNER_MAP.get(f"WC-{r['year']}","") == r["team_name"]), axis=1)
    return targets

targets = compute_targets(m_team_app)

elo_wc = (elo_df[elo_df["year"].isin(WC_YEARS)]
          .rename(columns={"team":"team_name","elo":"elo_pre_tournament"}))

team_features = (
    targets
    .merge(elo_wc,     on=["year","team_name"], how="left")
    .merge(form_df,    on=["year","team_name"], how="left")
    .merge(squad_df,   on=["year","team_name"], how="left")
    .merge(history_df, on=["year","team_name"], how="left")
)

fill_zero = ["elo_pre_tournament","form_win_rate","form_draw_rate","form_gd_per_game",
             "form_gf_per_game","form_ga_per_game","form_points_per_game",
             "squad_avg_exp","wc_appearances","best_stage_ever","is_host","won_last_wc"]
for col in fill_zero:
    if col in team_features.columns:
        team_features[col] = team_features[col].fillna(0)

if "squad_avg_age" in team_features.columns:
    team_features["squad_avg_age"] = team_features["squad_avg_age"].fillna(
        team_features["squad_avg_age"].median())

team_features = team_features.sort_values(["year","team_name"]).reset_index(drop=True)

print(f"team_features: {team_features.shape[0]:,} rows x {team_features.shape[1]} cols")
nulls = team_features.isnull().sum()
print(f"Nulls: {nulls[nulls>0].to_dict() if nulls.any() else 'none'}")
display(team_features[team_features["year"]==2022]
        .sort_values("elo_pre_tournament", ascending=False)
        .head(8)[["team_name","elo_pre_tournament","form_points_per_game",
                   "squad_avg_age","squad_avg_exp","wc_appearances","stage_rank"]])

## 11. Player Features → player_features

In [ ]:
goal_counts = (
    m_goals[~m_goals["own_goal"].astype(bool)]
    .assign(year=lambda d: d["year"].astype(int))
    .groupby(["year","player_id","team_name"]).agg(
        goals_scored  = ("goal_id", "count"),
        penalty_goals = ("penalty", "sum"),
    ).reset_index()
)
goal_counts["open_play_goals"] = goal_counts["goals_scored"] - goal_counts["penalty_goals"]

app_counts = (
    m_player_app
    .assign(year=lambda d: d["year"].astype(int))
    .groupby(["year","player_id","team_name"]).agg(
        matches_played  = ("match_id", "count"),
        matches_started = ("starter",  "sum"),
        position_name   = ("position_name", "first"),
    ).reset_index()
)

bio = wc_players[["player_id","family_name","given_name","birth_date","list_tournaments"]].copy()
bio["player_name"] = bio["given_name"].str.strip() + " " + bio["family_name"].str.strip()
bio["birth_date"]  = pd.to_datetime(bio["birth_date"], errors="coerce")

t_starts = (
    wc_tournaments[wc_tournaments["tournament_name"].str.contains("Men", na=False)]
    .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
    .set_index("year")["start_date"].to_dict()
)

player_features = (
    app_counts
    .merge(goal_counts, on=["year","player_id","team_name"], how="left")
    .merge(bio, on="player_id", how="left")
)

player_features["goals_scored"]    = player_features["goals_scored"].fillna(0).astype(int)
player_features["penalty_goals"]   = player_features["penalty_goals"].fillna(0).astype(int)
player_features["open_play_goals"] = player_features["open_play_goals"].fillna(0).astype(int)

player_features["t_start"] = player_features["year"].map(t_starts)
player_features["age"] = (
    (player_features["t_start"] - player_features["birth_date"]).dt.days / 365.25
).round(1)

def prev_wc(row):
    try:
        yrs = [int(y.strip()) for y in str(row["list_tournaments"]).split(",")]
        return sum(1 for y in yrs if y < int(row["year"]))
    except: return 0
player_features["wc_exp"] = player_features.apply(prev_wc, axis=1)

for col in ["won_golden_boot","won_best_player","won_young_player"]:
    player_features[col] = False

for _, row in m_awards.iterrows():
    col = AWARD_IDS.get(row["award_id"])
    if not col: continue
    mask = ((player_features["year"] == int(row["year"])) &
            (player_features["player_id"] == row["player_id"]))
    player_features.loc[mask, col] = True

player_features = player_features.drop(
    columns=["t_start","birth_date","list_tournaments"], errors="ignore"
).reset_index(drop=True)

print(f"player_features: {player_features.shape[0]:,} rows x {player_features.shape[1]} cols")
print(f"  Golden Boot flags : {player_features['won_golden_boot'].sum()}")
print(f"  Best Player flags : {player_features['won_best_player'].sum()}")
print(f"  Young Player flags: {player_features['won_young_player'].sum()}")

## 12. Feature Analysis

In [ ]:
FEAT_COLS = [c for c in [
    "elo_pre_tournament","form_win_rate","form_gd_per_game","form_points_per_game",
    "form_gf_per_game","squad_avg_age","squad_avg_exp",
    "wc_appearances","best_stage_ever","is_host","won_last_wc"
] if c in team_features.columns]

corr = (team_features[FEAT_COLS + ["stage_rank"]]
        .corr()["stage_rank"].drop("stage_rank")
        .sort_values(ascending=False))
print("Correlation with stage_rank:")
print(corr.round(3).to_string())
print("\nTop features (|corr| > 0.2) are reliable predictors.")
print("Features with |corr| < 0.05 add noise — consider dropping.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Feature Distributions — Winners vs Others (v2: full international data)", fontsize=13)

plot_feats = [
    ("elo_pre_tournament",    "Elo Rating (pre-tournament)"),
    ("form_points_per_game",  "Form: Points/Game (last 18mo)"),
    ("form_gd_per_game",      "Form: Goal Diff/Game"),
    ("squad_avg_age",         "Squad Average Age"),
    ("squad_avg_exp",         "Squad WC Experience"),
    ("wc_appearances",        "WC Appearances"),
]

for ax, (col, title) in zip(axes.flat, plot_feats):
    if col not in team_features.columns:
        ax.set_visible(False); continue
    for won, grp in team_features.groupby("won_tournament"):
        ax.hist(grp[col].dropna(), bins=20, alpha=0.7, edgecolor="white",
                color="#FFD700" if won else "#4A90D9",
                label="Winner" if won else "Other", density=True)
    ax.set_title(title); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 13. Save Outputs

In [ ]:
from google.colab import files as colab_files

team_features.to_csv("team_features.csv", index=False)
player_features.to_csv("player_features.csv", index=False)

colab_files.download("team_features.csv")
colab_files.download("player_features.csv")

print(f"team_features.csv   : {team_features.shape[0]:,} rows x {team_features.shape[1]} cols")
print(f"player_features.csv : {player_features.shape[0]:,} rows x {player_features.shape[1]} cols")
print("\nThese files are the input to the modelling notebook.")